In [1]:
import os
import sys
import json

import pandas as pd
import numpy as np

from openai import AsyncOpenAI

from dotenv import load_dotenv
load_dotenv()

sys.path.append(os.path.dirname(os.path.abspath(os.getcwd())))
from src.prompts import SYSTEM_MSG, TEMPLATE, INSTRUCTION, INPUT_STR

In [2]:
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
lstm_output = "LSTM model output : +6.25%"
bert_output = "BERT model output : 30% Positive, 60% Neutral, 10% Negative"
macro_data = "Inflation Rate Higher than normal"
open_price = 312
cash = 30043
shares_owned = 30

input_str = INPUT_STR.format(
    lstm_output=lstm_output,
    bert_output=bert_output,
    macro_data=macro_data,
    open_price=open_price,
    cash=cash,
    shares_owned=shares_owned
)

instruction = INSTRUCTION.format(
    input_str=input_str,
)

prompt = TEMPLATE.format(
    instruction=instruction,
)

response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
            response_format={"type": "json_object"},
        )

print(response)

ChatCompletion(id='chatcmpl-DNPredjiOR5PRUQSF3yS362TJ3Ss8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n    "action": "BUY",\n    "share": "10",\n    "reason": "The LSTM model predicts a positive return, and the overall sentiment is mostly neutral, indicating potential stability."\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774473338, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_c502f8cb2c', usage=CompletionUsage(completion_tokens=42, prompt_tokens=342, total_tokens=384, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [10]:
result_json = json.loads(response.choices[0].message.content)

In [11]:
action = result_json.get("action", "SELL")
share = result_json.get("share", "30")
reason = result_json.get("reason", "-")